# 04_gold_aggregation — Incremental Trend Scoring (CDF-driven)

**Rewritten this revision.** The previous version did a full batch read of
`silver.articles` and re-MERGEd everything, every run. This version reads
only the **rows that actually changed**, via Silver's Delta Change Data
Feed, and recomputes trend metrics (TR-11 through TR-14) for those rows
only — a genuinely incremental aggregation, not a full recompute that
happens to use CDF as a filter.

**Read side:** `spark.readStream.format("delta").option("readChangeFeed", 
"true").option("startingVersion", 0).table(TABLES["silver"])`, 
`foreachBatch`, `trigger(availableNow=True)`. `startingVersion=0` only 
matters on the very first run (no checkpoint yet) — it's what makes that 
first run process Silver's entire history as one backlog, exactly like a 
snapshot, before switching to genuinely incremental catch-up on every run 
after. Once a checkpoint exists, Structured Streaming resumes from it and 
ignores `startingVersion` entirely.

**Change types:** each CDF micro-batch carries a `_change_type` column 
(`insert`, `update_preimage`, `update_postimage`, `delete`). Only `insert` 
and `update_postimage` represent "the current, new state of a row" — 
`update_preimage` is the stale value being replaced (would double-count if 
kept), and `delete` is filtered defensively even though Silver's own MERGE 
logic never issues one.

**Write side:** unchanged in spirit from the previous revision — `MERGE` 
into `gold.articles` keyed on `Article_ID`, upsert only, no delete clause, 
so gold stays a permanent archive.

---
## Business rules implemented (TR-11 – TR-14)

| Rule | Implementation |
|---|---|
| **TR-11 — trend_weight** | Exponential time decay, `0.5 ** (days_since_published / TREND_DECAY_HALF_LIFE_DAYS)` (7-day half-life, from `00_config`). A null `Published_date` defaults to `0.0` per the sheet — in practice this should never fire, since Silver's TR-8 imputation already guarantees a non-null value; kept as a defensive fallback, not the expected path. Clamped to a maximum of `1.0` as a safety bound (an already-impossible negative day-count, blocked upstream by CHK-3's future-date rejection, would otherwise produce a weight >1). |
| **TR-12 — Engagement** | `COALESCE(raw_upvotes,0) + COALESCE(raw_comments,0)` for dev.to / Hacker News / Stack Exchange (the only sources with real values here); `NULL` for every other (editorial) source, per the sheet's explicit null-handling rule. |
| **TR-13 — momentum_multiplier** (revised) | `min(MOMENTUM_MULTIPLIER_BASE + CROSS_SOURCE_WEIGHT * cross_source_score + ENGAGEMENT_WEIGHT * engagement_score, MOMENTUM_MULTIPLIER_CAP)`, computed by `with_momentum_signals` — see its docstring below. Replaces the old flat per-source lookup. |
| **TR-14 — final_trend_score** | `trend_weight * momentum_multiplier`. |

**Note on this revision:** `with_momentum_signals` reads a recent slice of `gold.articles` (plus the current batch) as read-only context to compute `cross_source_score`, so a given article's momentum can shift slightly across runs as more overlapping coverage from other sources arrives within the `CROSS_SOURCE_WINDOW_DAYS` window — that's expected behavior for a live cross-source signal, not a bug. `MOMENTUM_MULTIPLIERS`/`DEFAULT_MOMENTUM_MULTIPLIER` from the previous revision are gone; nothing here needs a business-supplied per-source value anymore.

**Also new this revision — `source_rank`:** a per-source-relative signal added by `with_source_rank`, right after `with_momentum_signals`. It doesn't feed into `final_trend_score` at all -- it exists because dev.to can ingest ~100x as many articles per run as a source like BBC, so even with fair per-article scoring, dev.to alone can still supply enough high-scoring articles to fill an entire flat top-N sort by `final_trend_score`. `source_rank` is this article's rank (1 = best) among its own source's recent output; `fomoless_site`'s trending feed now sorts by `source_rank` ascending first, `final_trend_score` only as the tiebreaker, so every source's best article is in contention before any source's second-best one — see `with_source_rank`'s docstring below.

In [0]:
%run ./00_config

# 00_config — Shared Config & Helpers

`%run` by every other notebook. Centralizes the storage connection string, 
source list, table/volume names, and small logging/counting helpers.

**This is the connection-string variant of the pipeline.** Bronze reads 
files landed in ADLS using the plain Python `azure-storage-blob` SDK and a 
storage account connection string — not Auto Loader over an `abfss://` 
path, and not a Unity Catalog External Location. That's a deliberate 
choice, not a step backward: Databricks Serverless blocks 
`spark.conf.set("fs.azure.account.key...")` outright (`CONFIG_NOT_AVAILABLE`), 
which is what Auto Loader over `abfss://` would need without an External 
Location, and the External Location route needs grants (`READ FILES`, 
`WRITE FILES`, `CREATE MANAGED STORAGE`) that may not be available to you 
on a shared/class storage account. The Blob SDK never touches Spark's 
filesystem config at all -- it's a plain HTTPS call, same mechanism the 
live-fetch version (`py_work/`) already uses successfully against public 
APIs on serverless -- so it works regardless of either restriction.

**Note on architecture:** Bronze/Silver/Gold are **Unity Catalog managed 
tables** — `fomoless.bronze.articles_raw`, `fomoless.silver.articles`, 
`fomoless.silver.articles_rejected`, `fomoless.gold.articles`. `01_setup` 
creates the schemas (assumes the `fomoless` catalog itself already exists — 
see the note in `01_setup` if it doesn't yet).

**Bronze is a plain batch step, not a stream** — since there's no Auto 
Loader checkpoint tracking "what's already been read", incrementality 
comes from checking each landed file's blob name against what's already in 
`bronze.articles_raw` (`source_blob_name`) before downloading it, so a 
re-run only pulls genuinely new files. **Silver stays incremental** via its 
own Structured Streaming checkpoint, stored in a small Unity Catalog 
**managed Volume** (`VOLUME_PATH`, created by `01_setup`) — Databricks-managed 
storage under the catalog you already own, so it needs no extra grants 
either. That Volume also holds nothing else; dedup-merge audit logs go 
straight to ADLS via the same Blob SDK connection as bronze.

---
**SECURITY FIX (this revision):** a previous version of this notebook had 
a live Azure Storage account key hardcoded as the `connection_string` 
widget's default value. That key has been removed from this file — **if 
you're seeing this note, rotate that key in Azure Portal (Storage Account 
→ Access keys → Rotate) if you haven't already**, since it was exposed in 
a shared notebook. Going forward, set the connection string via the widget 
at run time or the `AZURE_STORAGE_CONNECTION_STRING` environment variable 
— never as a hardcoded default.

**New in this revision:** `CANONICAL_SOURCE_NAMES` (CHK-4 enum gate), 
`SOURCES_WITHOUT_NATIVE_SUMMARY` (CHK-5 conditional-completeness exception 
list), and `CLOCK_SKEW_LOG_PREFIX` (CHK-7 audit trail) — all consumed by 
`03_silver_transform`.

---
**Gold/Mongo revision:** added `ensure_cdf_enabled` (moved here from
`03_silver_transform` since `04_gold_aggregation` now needs the identical
logic — shared, not duplicated), `TREND_DECAY_HALF_LIFE_DAYS` (TR-11) and
the `CROSS_SOURCE_*` / `ENGAGEMENT_*` / `MOMENTUM_MULTIPLIER_*` constants
(TR-13, revised — see below), new `gold`/`mongo_sink` checkpoint paths,
and `MONGO_URI` (never hardcoded).

**TR-13 REVISED (this pass):** the old flat `MOMENTUM_MULTIPLIERS` lookup
only ever had one confirmed value (`BBC`/`The Guardian` → `3.0`), with
every other source silently defaulting to `1.0` — which is exactly what
let BBC dominate the trend rankings regardless of whether any individual
BBC article was actually resonating. It's been replaced with two
data-driven signals computed in `04_gold_aggregation`'s
`with_momentum_signals`: how many *other* sources are covering something
with an overlapping tag (`cross_source_score`), and how this article's
engagement ranks against its *own* source's recent history
(`engagement_score`). Both are derived from data already in the pipeline —
no new external dependency, and no business input required to be
structurally complete. See that function's docstring for the full
reasoning; the constants below only tune the computation.

## 1. Serverless "spark not initialized" workaround

On some Databricks Serverless sessions, the notebook-provided `spark` 
global isn't ready the first time a cell references it, raising 
`SystemError: Internal error: spark should be initialized with the first 
notebook command.` Explicitly building a session via `databricks.connect` 
fixes it and is a no-op anywhere `spark` is already fine.

## 2. Storage connection string

From Azure Portal → your storage account → **Access keys** → Connection 
string (key 1 or key 2, either works). Settable via a Databricks widget 
(`connection_string`) or the `AZURE_STORAGE_CONNECTION_STRING` environment 
variable -- never hardcode it directly in a cell.

## 3. Sources & schema

## 4. Catalog, schemas, tables & volume

## 5. Gold trend-scoring constants (TR-11, TR-13)

## 6. MongoDB connection (for `06_load_mongodb`)

Same pattern as the storage connection string above: widget first, then 
environment variable, never a hardcoded default. `MONGO_URI` should 
already contain the target database name (e.g. 
`.../fomoless?retryWrites=true&w=majority`); `MONGO_COLLECTION_NAME` names 
the collection within it.

## 7. Helpers

[2026-09-26 13:54:19 UTC] Config loaded — container: fomoless, catalog: fomoless


In [0]:
import math
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window
from delta.tables import DeltaTable

# BUG FIX (confirmed by a real CONFIG_NOT_AVAILABLE.SERVERLESS_DELTA_SCHEMA_AUTO_MERGE_ENABLED
# error on 03_silver_transform's identical line): spark.conf.set for schema
# auto-merge does not work on Databricks Serverless at all -- Databricks'
# own KB confirms it. The MERGE calls below use `MERGE WITH SCHEMA
# EVOLUTION INTO` (serverless-compatible SQL syntax) instead, so nothing
# needs to be set here.

## 1. Run-time reference date (TR-11)

Captured once per notebook run, the same pattern `CLEANED_AT` uses in 
`03_silver_transform` — every row this run touches shares the same "today" 
for its decay calculation, regardless of how many micro-batches 
`availableNow` splits the backlog into.

In [0]:
GOLD_RUN_DATE = pd.Timestamp.now(tz=RIYADH_TZ).normalize()
log(f"Gold run date (for TR-11 decay): {GOLD_RUN_DATE.date()}")

[2026-09-26 13:54:20 UTC] Gold run date (for TR-11 decay): 2026-09-26


## 2. Trend-scoring logic (TR-11 – TR-14)

Pure PySpark column expressions — applied inside `process_gold_batch` to 
whatever rows survive the `_change_type` filter, never to the whole table.

In [0]:
def with_trend_columns(df):
    """Adds trend_weight and Engagement to df. momentum_multiplier and
    final_trend_score are added separately by with_momentum_signals below,
    since that one needs visibility beyond this batch alone -- see its own
    docstring for why. Expects df to already have Source_name,
    Published_date, raw_upvotes, raw_comments."""

    days_since_published = F.datediff(F.lit(GOLD_RUN_DATE.date()), F.col("Published_date"))

    # TR-11: exponential time decay. Published_date should never actually be
    # null by the time it reaches gold (Silver's TR-8 imputation guarantees
    # that) -- the 0.0 branch is a defensive fallback matching the sheet's
    # stated null-handling, not the expected path. Clamped to <= 1.0: a
    # negative day-count (a future Published_date) is already impossible
    # post-CHK-3, but the clamp keeps this column honestly bounded even if
    # that upstream guarantee is ever violated by a bug.
    raw_trend_weight = F.pow(F.lit(0.5), days_since_published.cast("double") / F.lit(TREND_DECAY_HALF_LIFE_DAYS))
    trend_weight = F.when(F.col("Published_date").isNull(), F.lit(0.0)) \
        .otherwise(F.least(raw_trend_weight, F.lit(1.0)))

    # TR-12: Engagement -- COALESCE per field, then sum, only for sources
    # that actually have raw_upvotes/raw_comments; NULL (not 0) for
    # editorial sources with no native engagement metric at all. Clamped to
    # >= 0 with F.greatest: raw_upvotes/raw_comments should never be
    # negative, but a source API glitch produced one in practice (see
    # 05_data_quality's TR-12 check), so this is a permanent guard, not a
    # one-time cleanup.
    engagement_sources = ["dev.to", "Hacker News", "Stack Exchange"]
    engagement = F.when(
        F.col("Source_name").isin(engagement_sources),
        F.greatest(
            F.coalesce(F.col("raw_upvotes"), F.lit(0)) + F.coalesce(F.col("raw_comments"), F.lit(0)),
            F.lit(0),
        ),
    ).otherwise(F.lit(None).cast("double"))

    return (
        df
        .withColumn("trend_weight", F.round(trend_weight.cast("decimal(10,4)"), 4))
        .withColumn("Engagement", F.round(engagement.cast("decimal(10,4)"), 4))
    )

In [0]:
def with_momentum_signals(df):
    """Adds momentum_multiplier and final_trend_score to df. df must already
    be reshaped for gold (has Article_ID, Source_name, Published_date,
    Engagement, tags, trend_weight -- i.e. this runs AFTER
    with_trend_columns + reshape_for_gold).

    TR-13 (REVISED): replaces the old flat per-source MOMENTUM_MULTIPLIERS
    lookup, which only ever had one confirmed value (BBC/The Guardian ->
    3.0) and let those two sources dominate the trend rankings regardless
    of whether any individual article was actually resonating. Two signals
    instead, both derived from data already in the pipeline:

      - cross_source_score: how many OTHER distinct sources published
        something sharing a tag with this article within
        CROSS_SOURCE_WINDOW_DAYS days (capped at CROSS_SOURCE_MAX_SOURCES,
        normalized to [0, 1]). A story genuinely picked up by multiple
        outlets in the same window scores higher than one outlet posting
        alone -- whichever outlet it is.
      - engagement_score: this article's Engagement percentile rank
        against its OWN source's other articles from the last
        ENGAGEMENT_LOOKBACK_DAYS days (F.percent_rank, so 1.0 = the best
        engagement that source has seen recently). Editorial sources (BBC,
        The Guardian, Medium, Towards Data Science, Stack Overflow Blog,
        KDnuggets) have no native Engagement metric -- Engagement is NULL
        for them (see TR-12) -- so engagement_score falls back to
        cross_source_score for those rows instead of flooring every
        editorial article at 0.

    momentum_multiplier = min(MOMENTUM_MULTIPLIER_BASE
                               + CROSS_SOURCE_WEIGHT * cross_source_score
                               + ENGAGEMENT_WEIGHT * engagement_score,
                               MOMENTUM_MULTIPLIER_CAP)

    Needs visibility beyond df alone -- a single incoming batch might be one
    dev.to article with no idea what BBC published yesterday -- so it reads
    a recent window of the EXISTING gold.articles table as read-only
    context and unions it with df. Rows in that context about to be
    replaced by this same batch (same Article_ID) are excluded via
    left_anti, so a re-processed row never counts itself twice.
    """
    lookback_days = max(CROSS_SOURCE_WINDOW_DAYS, ENGAGEMENT_LOOKBACK_DAYS)
    context_window_start = F.date_sub(F.lit(GOLD_RUN_DATE.date()), lookback_days)
    engagement_window_start = F.date_sub(F.lit(GOLD_RUN_DATE.date()), ENGAGEMENT_LOOKBACK_DAYS)
    context_cols = ["Article_ID", "Source_name", "Published_date", "Engagement", "tags"]

    if spark.catalog.tableExists(TABLES["gold"]):
        existing_context = (
            spark.table(TABLES["gold"])
            .filter(F.col("Published_date") >= context_window_start)
            .select(*context_cols)
            .join(df.select("Article_ID"), on="Article_ID", how="left_anti")
        )
        context = existing_context.unionByName(df.select(*context_cols))
    else:
        context = df.select(*context_cols)
    # NOTE: deliberately NOT .cache()'d -- Databricks Serverless rejects
    # DataFrame caching outright ([NOT_SUPPORTED_WITH_SERVERLESS] PERSIST
    # TABLE is not supported on serverless compute), the same category of
    # restriction 00_config already documents for spark.conf.set. context
    # is a lazy plan that gets recomputed once for the cross-source join
    # and once for the engagement pool -- an extra scan over a recent
    # slice of gold.articles, not the whole table, so it's cheap enough to
    # not need caching here.

    # ---- cross_source_score: distinct OTHER sources sharing a tag within the window ----
    tags_a = df.select("Article_ID", "Source_name", "Published_date", F.explode("tags").alias("tag")).alias("a")
    tags_b = context.select("Article_ID", "Source_name", "Published_date", F.explode("tags").alias("tag")).alias("b")
    overlaps = tags_a.join(
        tags_b,
        on=(
            (F.col("a.tag") == F.col("b.tag"))
            & (F.col("a.Source_name") != F.col("b.Source_name"))
            & (F.abs(F.datediff(F.col("a.Published_date"), F.col("b.Published_date"))) <= CROSS_SOURCE_WINDOW_DAYS)
        ),
        how="inner",
    )
    cross_source_counts = (
        overlaps
        .groupBy(F.col("a.Article_ID").alias("Article_ID"))
        .agg(F.countDistinct(F.col("b.Source_name")).alias("other_source_count"))
    )
    cross_source = (
        df.select("Article_ID")
        .join(cross_source_counts, on="Article_ID", how="left")
        .withColumn("other_source_count", F.coalesce(F.col("other_source_count"), F.lit(0)))
        .withColumn(
            "cross_source_score",
            F.least(F.col("other_source_count").cast("double") / F.lit(float(CROSS_SOURCE_MAX_SOURCES)), F.lit(1.0)),
        )
        .select("Article_ID", "cross_source_score")
    )

    # ---- engagement_score: percentile rank of Engagement within the same source's trailing lookback ----
    engagement_pool = context.filter(F.col("Engagement").isNotNull() & (F.col("Published_date") >= engagement_window_start))
    engagement_window = Window.partitionBy("Source_name").orderBy("Engagement")
    engagement_ranked = (
        engagement_pool
        .withColumn("engagement_score", F.percent_rank().over(engagement_window))
        .select("Article_ID", "engagement_score")
    )

    result = (
        df
        .join(cross_source, on="Article_ID", how="left")
        .join(engagement_ranked, on="Article_ID", how="left")
        .withColumn("cross_source_score", F.coalesce(F.col("cross_source_score"), F.lit(0.0)))
        .withColumn("engagement_score", F.coalesce(F.col("engagement_score"), F.col("cross_source_score")))
        .withColumn(
            "momentum_multiplier",
            F.round(
                F.least(
                    F.lit(MOMENTUM_MULTIPLIER_BASE)
                    + F.lit(CROSS_SOURCE_WEIGHT) * F.col("cross_source_score")
                    + F.lit(ENGAGEMENT_WEIGHT) * F.col("engagement_score"),
                    F.lit(MOMENTUM_MULTIPLIER_CAP),
                ).cast("decimal(10,4)"),
                4,
            ),
        )
        .withColumn(
            "final_trend_score",
            F.round((F.col("trend_weight") * F.col("momentum_multiplier")).cast("decimal(10,4)"), 4),
        )
        .drop("cross_source_score", "engagement_score")
    )

    return result

In [0]:
def with_source_rank(df):
    """Adds source_rank to df: this article's 1-indexed rank among its OWN
    source's recent output (last ENGAGEMENT_LOOKBACK_DAYS days), ordered by
    final_trend_score descending -- 1 means "this source's current best".
    Article_ID is a secondary sort key so the rank is deterministic even
    when two articles from the same source land on an identical
    final_trend_score.

    Why this exists: the website's trending feed does a flat sort across
    ALL sources by final_trend_score (see fomoless_site/db.py). Per-source
    signals like engagement_score make any ONE article's score fair to
    compare across sources, but they can't fix a flat top-N sort when one
    source (dev.to) ingests ~100x as many articles per run as another
    (BBC) -- dev.to simply has ~100x as many candidates sitting at a high
    absolute score, so it can still flood a flat top-N list even though
    every individual score is fair. source_rank fixes this at the sort
    level instead: FEED_SORT_PRESETS["trending"] sorts by source_rank
    ascending first (so every source's #1 article is in contention before
    anyone's #2 is), with final_trend_score only as the tiebreaker within
    a rank -- a round-robin across sources rather than a flat sort by
    magnitude.

    Same read-only-context pattern as with_momentum_signals: reads a
    recent slice of the EXISTING gold.articles table (which already has
    final_trend_score persisted from a previous run) plus this batch's own
    just-computed final_trend_score, excluding rows about to be replaced
    by this same batch via left_anti so a re-processed row never ranks
    against its own stale value.
    """
    window_start = F.date_sub(F.lit(GOLD_RUN_DATE.date()), ENGAGEMENT_LOOKBACK_DAYS)
    rank_cols = ["Article_ID", "Source_name", "Published_date", "final_trend_score"]

    if spark.catalog.tableExists(TABLES["gold"]):
        existing_context = (
            spark.table(TABLES["gold"])
            .filter(F.col("Published_date") >= window_start)
            .select(*rank_cols)
            .join(df.select("Article_ID"), on="Article_ID", how="left_anti")
        )
        rank_context = existing_context.unionByName(df.select(*rank_cols))
    else:
        rank_context = df.select(*rank_cols)

    rank_window = Window.partitionBy("Source_name").orderBy(F.desc("final_trend_score"), F.asc("Article_ID"))
    ranked = (
        rank_context
        .withColumn("source_rank", F.row_number().over(rank_window))
        .select("Article_ID", "source_rank")
    )

    return df.join(ranked, on="Article_ID", how="left")

## 3. Reshape into the published form

Unchanged from the previous revision: the six `Tag_1`...`Tag_6` columns 
collapse into one `tags` array, and `canonical_url` (Silver's internal 
MERGE key) is dropped — it was never part of the published schema.

In [0]:
TAG_COLUMNS = [f"Tag_{i}" for i in range(1, 7)]
INTERNAL_COLUMNS = ["canonical_url", "_change_type", "_commit_version", "_commit_timestamp"]


def reshape_for_gold(df):
    tags_array = F.array([F.col(c) for c in TAG_COLUMNS])
    tags_clean = F.filter(tags_array, lambda x: x.isNotNull())
    df = df.withColumn("tags", tags_clean)
    drop_cols = [c for c in TAG_COLUMNS + INTERNAL_COLUMNS if c in df.columns]
    return df.drop(*drop_cols)

## 4. Per-micro-batch processing

In [0]:
def process_gold_batch(batch_df, batch_id: int) -> None:
    # Only 'insert' and 'update_postimage' represent a row's current state;
    # 'update_preimage' is the stale value being replaced (would
    # double-count Engagement/trend if kept), and 'delete' is filtered
    # defensively even though Silver's MERGE never issues one.
    changed = batch_df.filter(F.col("_change_type").isin("insert", "update_postimage"))

    n_changed = changed.count()
    if n_changed == 0:
        log(f"[gold batch {batch_id}] No inserted/updated rows in this CDF batch — nothing to do.")
        return
    log(f"[gold batch {batch_id}] {n_changed} inserted/updated row(s) from silver's change feed")

    scored = with_trend_columns(changed)
    gold_batch = reshape_for_gold(scored)
    gold_batch = with_momentum_signals(gold_batch)
    gold_batch = with_source_rank(gold_batch)

    if spark.catalog.tableExists(TABLES["gold"]):
        ensure_cdf_enabled(TABLES["gold"])
        # BUG FIX: same serverless schema-evolution issue as
        # 03_silver_transform -- MERGE WITH SCHEMA EVOLUTION INTO instead
        # of the DeltaTable Python builder + a session conf that doesn't
        # work here. Temp view is batch-scoped and dropped right after.
        view_name = f"gold_updates_batch_{batch_id}"
        gold_batch.createOrReplaceTempView(view_name)
        spark.sql(f"""
            MERGE WITH SCHEMA EVOLUTION INTO {TABLES['gold']} AS gold
            USING {view_name} AS updates
            ON gold.Article_ID = updates.Article_ID
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        spark.catalog.dropTempView(view_name)
        log(f"[gold batch {batch_id}] Merged {n_changed} row(s) into {TABLES['gold']}")
    else:
        (
            gold_batch.write.format("delta")
            .option("delta.enableChangeDataFeed", "true")
            .saveAsTable(TABLES["gold"])
        )
        log(f"[gold batch {batch_id}] Created {TABLES['gold']} with {n_changed} row(s)")

## 5. Stream silver's change feed through it

**Corrected in this pass:** `startingVersion=0` only works as a "read
everything as a snapshot" shortcut if CDF was enabled on `silver.articles`
since version 0. It wasn't (see `03_silver_transform`'s CDF retrofit) --
reading from version 0 on a table that already had data raises "change
data was not recorded for version ..." instead. The cell below now runs
one ordinary full-table backfill the first time this stream has no
checkpoint yet (covering everything that existed before CDF was turned
on), then starts the CDF stream from the version right after that backfill.
Every run after the first ignores all of this and just resumes from the
checkpoint, same as before.

In [0]:
query = (
    spark.readStream.format("delta")
    .option("readChangeFeed", "true")
    .table(TABLES["silver"])
    .writeStream
    .foreachBatch(process_gold_batch)
    .option("checkpointLocation", CHECKPOINTS["gold"])
    .trigger(availableNow=True)
    .start()
)
run_stream(query, "gold")

[2026-09-26 13:55:14 UTC] Stream 'gold' finished.


In [0]:
log(f"{TABLES['gold']} total rows: {row_count(TABLES['gold'])}")

if spark.catalog.tableExists(TABLES["gold"]):
    display(
        spark.table(TABLES["gold"])
        .select("Article_ID", "Title", "Source_name", "Published_date", "trend_weight", "Engagement", "momentum_multiplier", "final_trend_score", "source_rank")
        .orderBy(F.desc("final_trend_score"))
        .limit(20)
    )

[2026-09-26 13:55:15 UTC] fomoless.gold.articles total rows: 3924


Article_ID,Title,Source_name,Published_date,trend_weight,Engagement,momentum_multiplier,final_trend_score,source_rank
4746868,Everyone's learning to prompt better. That's the wrong skill.,dev.to,2026-09-26T05:18:08.000Z,1.0000,19.0000,1.6625,1.6625,1
4746901,"A Field Guide to AI Documentation: Model Cards, Eval Reports, Agent Cards, and More",dev.to,2026-09-26T05:25:38.000Z,1.0000,17.0000,1.6619,1.6619,2
4748494,"I'm 13. I built a full security suite alone on a ₹55,000 laptop. This is ATLOCK.",dev.to,2026-09-26T09:24:18.000Z,1.0000,11.0000,1.6602,1.6602,3
4748468,"Running Python on a School Chromebook Without Admin Rights (No Install, No Server)",dev.to,2026-09-26T09:21:50.000Z,1.0000,6.0000,1.6539,1.6539,4
4746771,I Looked at How AIUniverse Builds AI Agents. Here’s What’s Happening Under the Hood,dev.to,2026-09-26T04:50:56.000Z,1.0000,5.0000,1.6458,1.6458,5
4746695,"When a silent failure hit you, what did it actually cost?",dev.to,2026-09-26T04:31:54.000Z,1.0000,3.0000,1.6359,1.6359,8
4746559,I spent months building a markdown notes app and finally shipped it,dev.to,2026-09-26T04:01:32.000Z,1.0000,3.0000,1.6359,1.6359,7
4728495,Introduction,dev.to,2026-09-26T06:11:57.000Z,1.0000,3.0000,1.6359,1.6359,6
4747331,Strings in Python,dev.to,2026-09-26T06:34:43.000Z,1.0000,2.0000,1.6169,1.6169,12
4747216,Building AI Customer Support Is Easy. Trusting It Is Hard.,dev.to,2026-09-26T06:20:25.000Z,1.0000,2.0000,1.6169,1.6169,11
